# Helmax PPE Detection — YOLOv8n Fine-tuning

In [ ]:
# Step 1: Install dependencies
!pip install ultralytics -q

In [ ]:
# Step 2: Upload your dataset zip to Kaggle and set this path
DATASET_PATH = '/kaggle/input/datasets/emmanuelselase/helmax-ppe/ppe'
DATA_YAML = f'{DATASET_PATH}/data.yaml'

In [ ]:
# Step 3: Fix paths in data.yaml for Kaggle environment
import yaml

with open(DATA_YAML, 'r') as f:
    config = yaml.safe_load(f)

config['train'] = f'{DATASET_PATH}/train/images'
config['val']   = f'{DATASET_PATH}/valid/images'
config['test']  = f'{DATASET_PATH}/test/images'

fixed_yaml_path = '/kaggle/working/data.yaml'
with open(fixed_yaml_path, 'w') as f:
    yaml.dump(config, f)

print('data.yaml fixed and written to', fixed_yaml_path)

In [ ]:
# Step 4: Train
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

model.train(
    data=fixed_yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    name='helmax_ppe',
    project='/kaggle/working/runs',
    patience=10,
    device=0
)

In [ ]:
# Step 5: Validate
model.val()

In [ ]:
# Step 6: Export to TFLite (int8 quantized for mobile)
best_model = YOLO('/kaggle/working/runs/helmax_ppe/weights/best.pt')

best_model.export(
    format='tflite',
    imgsz=640,
    int8=True
)

In [ ]:
# Step 7: Confirm output files
import os
weights_dir = '/kaggle/working/runs/helmax_ppe/weights'
for f in os.listdir(weights_dir):
    print(f)